In [4]:
!pip install transformers datasets torch

In [6]:
import pandas as pd
from datasets import Dataset

In [8]:
df = pd.read_csv('sentiment_yelp_data.csv')

In [10]:
df

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,sentiment,sentiment_label
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,wife took birthday breakfast excellent weather...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0,Positive,2
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,idea people give bad reviews place goes show p...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0,Positive,2
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love gyro plate rice good also dig candy selec...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0,Positive,2
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,rosie dakota love chaparral dog park convenien...,review,uZetl9T0NcROGOyFfughhg,1,2,0,Positive,2
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,general manager scott petello good egg go deta...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0,Positive,2
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,VY_tvNUCCXGXQeSvJl757Q,2012-07-28,Ubyfp2RSDYW0g7Mbr8N3iA,3,first visit lunch today used groupon ordered b...,review,_eqQoPtQ3e3UxLE4faT6ow,1,2,0,Positive,2
9996,EKzMHI1tip8rC1-ZAy64yg,2012-01-18,2XyIOQKbVFb6uXQdJ0RzlQ,4,called house deliciousness could go item item ...,review,ROru4uk5SaYc3rg8IU7SQw,0,0,0,Positive,2
9997,53YGfwmbW73JhFiemNeyzQ,2010-11-16,jyznYkIbpqVmlsZxSDSypA,4,recently visited olive ivy business last week ...,review,gGbN1aKQHMgfQZkqlsuwzg,0,0,0,Positive,2
9998,9SKdOoDHcFoxK5ZtsgHJoA,2012-12-02,5UKq9WQE1qQbJ0DJbc-B6Q,2,nephew moved scottsdale recently bunch friends...,review,0lyVoNazXa20WzUyZPLaQQ,0,0,0,Negative,0


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   business_id      10000 non-null  object
 1   date             10000 non-null  object
 2   review_id        10000 non-null  object
 3   stars            10000 non-null  int64 
 4   text             9999 non-null   object
 5   type             10000 non-null  object
 6   user_id          10000 non-null  object
 7   cool             10000 non-null  int64 
 8   useful           10000 non-null  int64 
 9   funny            10000 non-null  int64 
 10  sentiment        10000 non-null  object
 11  sentiment_label  10000 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 937.6+ KB


In [14]:
df.sentiment.unique()

array(['Positive', 'Negative', 'Neutral'], dtype=object)

In [16]:
print(df['text'].isnull().sum())  # Check for null values

1


In [18]:
print(df['text'].apply(lambda x: isinstance(x, str)).sum())  # Check for non-string entries

9999


In [20]:
# Drop rows where 'text' is null
df = df.dropna(subset=['text'])

# Ensure all values in the 'text' column are strings
df['text'] = df['text'].astype(str)

/var/folders/4w/nfkvhvrs0sxd55472ytdz0nh0000gp/T/ipykernel_55941/1682047218.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].astype(str)


In [22]:
# Convert the dataset to a Hugging Face `Dataset`
# Ensure the `sentiment` column is mapped to integers (0 for negative, 1 for positive)
df['sentiment'] = df['sentiment'].map({'Negative': 0, 'Neutral': 1, 'Positive': 2})
dataset = Dataset.from_pandas(df)

# Peek at the dataset to ensure it's loaded correctly
print(dataset)

Dataset({
    features: ['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id', 'cool', 'useful', 'funny', 'sentiment', 'sentiment_label', '__index_level_0__'],
    num_rows: 9999
})


/var/folders/4w/nfkvhvrs0sxd55472ytdz0nh0000gp/T/ipykernel_55941/699656763.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['sentiment'].map({'Negative': 0, 'Neutral': 1, 'Positive': 2})


In [24]:
#Tokenize data using BERT Tokenizer

from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize function to apply to each example
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Apply the tokenizer to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)


/Users/dzaytsev/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/9999 [00:00<?, ? examples/s]

In [25]:
#Prepare data for PyTorch Training

In [28]:
# Remove unnecessary columns
tokenized_dataset = tokenized_dataset.remove_columns(['business_id', 'date','review_id', 'stars', 'type', 'user_id', 'cool', 'useful', 'funny', 'sentiment_label'])

# Rename the sentiment_label column to labels
tokenized_dataset = tokenized_dataset.rename_column('sentiment', 'labels')

# Set the dataset format to PyTorch tensors
tokenized_dataset.set_format('torch')

In [30]:
#Split dataset to train and test


In [32]:
# Split the dataset into training and testing sets (80% train, 20% test)
train_test_split = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7999
})
Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [34]:
train_dataset['labels']

tensor([2, 2, 2,  ..., 2, 0, 2])

In [36]:
#Load the model

In [38]:
from transformers import BertForSequenceClassification

# Load pre-trained BERT for sequence classification (with 3 sentiment labels)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


/Users/dzaytsev/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/Users/dzaytsev/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/Users/dzaytsev/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classif

In [40]:
#Define training parameters and initialize trainer

In [ ]:
!pip install accelerate -U

In [72]:
from transformers import Trainer, TrainingArguments

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model
    evaluation_strategy="epoch",     # evaluate each epoch
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    num_train_epochs=5,              # number of epochs
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for logs
    logging_steps=10,
)

# Initialize the trainer
trainer = Trainer(
    model=model,                     # the pre-trained BERT model
    args=training_args,              # training arguments
    train_dataset=train_dataset,     # training dataset
    eval_dataset=test_dataset        # evaluation dataset
)

/Users/dzaytsev/anaconda3/lib/python3.11/site-packages/accelerate/accelerator.py:457: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None)
  warnings.warn(


In [73]:
#Train Model / Fine-Tune BERT

In [76]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.211300,0.287595
2,0.243400,0.293401
3,0.240000,0.237352
4,0.180200,0.251862
5,0.093200,0.286612


TrainOutput(global_step=2500, training_loss=0.24132169404029846, metrics={'train_runtime': 11245.1041, 'train_samples_per_second': 3.557, 'train_steps_per_second': 0.222, 'total_flos': 1.052322114203136e+16, 'train_loss': 0.24132169404029846, 'epoch': 5.0})

In [78]:
#Evaluate Model

In [80]:
# Evaluate the model
results = trainer.evaluate()
print(results)

{'eval_loss': 0.2866124212741852, 'eval_runtime': 58.1273, 'eval_samples_per_second': 34.407, 'eval_steps_per_second': 2.15, 'epoch': 5.0}


In [52]:
#Save Model

In [82]:
# Save the model and tokenizer
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')

('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/special_tokens_map.json',
 './fine_tuned_model/vocab.txt',
 './fine_tuned_model/added_tokens.json')

In [83]:
!zip mymodel.zip fine_tuned_model/

updating: fine_tuned_model/ (stored 0%)


In [86]:
#Using SavedModel

In [88]:
from transformers import BertTokenizer, BertForSequenceClassification

# Load the saved tokenizer
tokenizer = BertTokenizer.from_pretrained('./fine_tuned_model')

# Load the saved model
model = BertForSequenceClassification.from_pretrained('./fine_tuned_model')

# Example text to classify sentiment
text = "The movie was amazing and I loved it!"

# Tokenize the input text (same as how it was done during training)
inputs = tokenizer(text, return_tensors='pt', padding='max_length', truncation=True)

import torch

# Perform inference (get the logits)
outputs = model(**inputs)

# Extract the predicted label (index of the maximum value in logits)
predictions = torch.argmax(outputs.logits, dim=1)

# Map the prediction to the actual label (e.g., positive, negative, neutral)
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}  # Adjust according to your label mapping
predicted_label = label_map[predictions.item()]

print(f"Predicted sentiment: {predicted_label}")

Predicted sentiment: Positive


In [90]:
inputs = tokenizer("it was waste of time", return_tensors='pt', padding='max_length', truncation=True)

# Perform inference (get the logits)
outputs = model(**inputs)

# Extract the predicted label (index of the maximum value in logits)
predictions = torch.argmax(outputs.logits, dim=1)

# Map the prediction to the actual label (e.g., positive, negative, neutral)
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}  # Adjust according to your label mapping
predicted_label = label_map[predictions.item()]

print(f"Predicted sentiment: {predicted_label}")

Predicted sentiment: Negative
